In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use aws storage
del os.environ["BIRDDOG_USE_LOCAL_CACHE"]
if os.environ.get("BIRDDOG_USE_LOCAL_CACHE"):
    print("using local storage")
else:
    os.environ["BIRDDOG_AWS_ENVIRONMENT"] = "1"
    print("using aws storage")    

using aws storage


In [3]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [4]:
from birddog.tracker import (
    WikiDocTracker,
    WIKIMEDIA_COMMONS_DOC_TRACKER_SPEC,
    UK_WIKISOURCE_DOC_TRACKER_SPEC,
    )

from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater,
    )

2026-08-14 08:13:47,190 [INFO] Translation is enabled. Using GCP translator
2026-08-14 08:13:47,191 [INFO] Using Google Cloud translation API
2026-08-14 08:13:47,191 [INFO] GoogleCloudTranslator using REST API
2026-08-14 08:13:47,210 [INFO] Found credentials in environment variables.
2026-08-14 08:13:47,604 [INFO] Using AWS S3 bucket birddog-data for storage.
2026-08-14 08:13:47,937 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com


In [5]:
runtime = Runtime()

2026-08-14 08:13:52,889 [INFO] PageUpdateManager.init(): detect_environment==aws
2026-08-14 08:14:10,314 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-14 08:14:10,526 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     4.75    39.00       0.00           24
2026-08-14 08:14:10,848 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-14 08:14:12,453 [INFO] WikiDocTracker: base=https://commons.wikimedia.org, namespace=File (id=6)
2026-08-14 08:14:12,880 [INFO] WikiDocTracker: base=https://uk.wikisource.org, namespace=Файл (id=6)
2026-08-14 08:14:12,883 [INFO] KillSwitch: loading thresholds from resourc

In [29]:
tracker = WikiDocTracker(
    runtime,
    spec=UK_WIKISOURCE_DOC_TRACKER_SPEC,
)

2026-08-14 08:38:00,374 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   42.00     0.14    39.00       0.00           24
  uk.wikisource.org:api                  4.75     0.00     4.00       0.00            4
2026-08-14 08:38:00,647 [INFO] WikiDocTracker: base=https://uk.wikisource.org, namespace=Файл (id=6)


In [31]:
tracker._ensure_doc_map()
len(tracker._doc_map)

174455

In [32]:
title = "File:ДАЖО_67-8-13._1906-1907._Метрична_книга_лютеран_Емільчино.pdf"
key = tracker._kv_key(tracker._normalize_title(title))
print("cached:", key in tracker._doc_map)

cached: True


In [33]:
db = Database()

2026-08-14 08:41:52,672 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-14 08:41:52,896 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   43.00     0.00    39.00       0.00           24
  uk.wikisource.org:api                  4.75     0.00     4.00       0.00            4


In [34]:
live_hashes = set()
cursor = None
while True:
    print(cursor)
    batch, cursor = db.scan(
        "Documents", cursor=cursor, limit=1000,
        fields=["title", "url"])
    for rec in batch:
        url, title = rec.get("url", ""), rec.get("title")
        if title and url.startswith(tracker._base_url):
            live_hashes.add(tracker._kv_key(tracker._normalize_title(title)))
    if not cursor:
        break

stale = tracker._doc_map - live_hashes
print(f"{len(stale)} of {len(tracker._doc_map)} cached hashes have no matching live Documents row")

None
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
2026-08-14 08:42:54,567 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   49.00     1.77    39.00       0.00           24
  uk.wikisource.org:api                  4.

In [26]:
tracker._table_view

'BD:WDT'

In [19]:
batch, cursor = db.scan(
    "Documents", cursor=cursor, limit=1000,
    view_name=tracker._table_view, fields=["title", "url"])
    

2026-08-14 08:23:19,882 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   24.00     0.04    39.00       0.00           24
  uk.wikisource.org:api                  4.50     0.00     4.00       0.00            4


In [21]:
len(batch)

177

In [22]:
tracker._base_url

'https://commons.wikimedia.org'

In [27]:
len(live_hashes)

95

In [24]:
batch[:10]

[{'Id': 489338,
  'title': 'File:ЦДІАК_1158-1-89_Метрична_книга_єврейської_громади_м._Переяслав_про_шлюб_(1917–1918).pdf',
  'url': 'https://commons.wikimedia.org/wiki/File:ЦДІАК_1158-1-89_Метрична_книга_єврейської_громади_м._Переяслав_про_шлюб_(1917–1918).pdf'},
 {'Id': 489339,
  'title': 'Файл:ЦДІАК_1158-1-89_Метрична_книга_єврейської_громади_м._Переяслав_про_шлюб_(1917–1918).pdf',
  'url': 'https://uk.wikisource.org/wiki/File:ЦДІАК_1158-1-89_Метрична_книга_єврейської_громади_м._Переяслав_про_шлюб_(1917–1918).pdf'},
 {'Id': 489340,
  'title': 'File:ДАЧкО_Р-5625-1-5401_Кримінальна_справа_по_звинуваченню_Краснової-Гросман_Раїси_Самійлівни_по_статті..._(1937-1958).pdf',
  'url': 'https://commons.wikimedia.org/wiki/File:ДАЧкО_Р-5625-1-5401_Кримінальна_справа_по_звинуваченню_Краснової-Гросман_Раїси_Самійлівни_по_статті..._(1937-1958).pdf'},
 {'Id': 489341,
  'title': 'Файл:ДАКО_280-2-792._1850-1852_роки._Додаткові_ревізькі_казки_євреїв_Васильківського_і_Київського_повітів..pdf',
  'url': 

In [35]:
from birddog.wiki import get_recent_changes

In [36]:
c=get_recent_changes()

2026-08-14 08:48:06,004 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   60.00     0.22    39.00       0.00           24
  uk.wikisource.org:api                  5.00     0.00     4.00       0.00            4


In [38]:
len(c)

15555

In [39]:
c[0]

KeyError: 0

In [40]:
list(c.items())[0]

('Архів:ДАЧгО/Н-3001/1/696',
 {'timestamp': '2026-08-14T14:47:41Z', 'user': 'Boh.val', 'action': 'new'})